In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


# Layer 1- Analysis

In [ ]:
df = pd.read_csv("SpaceX_Falcon9.csv")
df

In [ ]:
df.info()

In [ ]:
df.describe()

# Step 3: Data Cleaning & Preprocessing


In [ ]:
df.isna()

In [ ]:
df.isna().sum()

In [ ]:
df['PayloadMass'] = df.groupby('BoosterVersion')['PayloadMass'].transform(lambda x: x.fillna(x.median()))
df      #using meaningfull groups to fill na values

In [ ]:
df.duplicated().sum() #Theres no duplicate rows

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], errors = 'coerce')

In [ ]:
#for encoding we will use Onehot encoding instead of label to make sure the Model is not biased
#lets copy the dataset to df2
df2 = df.copy()


In [ ]:
df2 = pd.get_dummies(df, columns = ['BoosterVersion','LaunchSite','Orbit'],drop_first = True)

In [ ]:
df2

In [ ]:
#Normalization- Some model may think Payloadmass since it has huge numbers it is more important than flight- so we will use normalization to shirnk the numbers mainating the orginal values and meaning.
#We will use Standardization since theres could be outliers in PayloadMass
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df2[['PayloadMass','Flights']] = scaler.fit_transform(df2[['PayloadMass','Flights']])
df2

In [ ]:
sns.histplot(df['PayloadMass'], kde=True)
plt.show() # this is right-skewed positve..

In [ ]:
sns.boxplot(x=df['PayloadMass'])
plt.show()

In [ ]:
sns.scatterplot(x='Flights', y='PayloadMass', data=df)
plt.show()

In [ ]:
sns.boxplot(x='Orbit', y='PayloadMass', data=df)
plt.xticks(rotation=45)
plt.show()

In [ ]:
sns.heatmap(df.select_dtypes(include='number').drop(columns=["Unnamed: 0"]).corr(), annot=True, cmap='coolwarm')
plt.show()


In [ ]:
sns.boxplot(x='Outcome', y='Flights', hue='BoosterVersion', data=df)
plt.xticks(rotation=45)
plt.show()

In [ ]:
sns.countplot(x='Outcome', data=df)
plt.xticks(rotation=45)

In [ ]:
sns.countplot(x='LaunchSite', data=df)


In [ ]:
df['Year'] = pd.to_datetime(df['Date']).dt.year
sns.countplot(x='Year', data=df)
plt.xticks(rotation=45)

In [ ]:
df['Year'] = pd.to_datetime(df['Date']).dt.year
df['Year'].value_counts().sort_index()

In [ ]:
#Lets see why 2019 had some drastic change, first lets see how payload correlates with counts of how many launches
sns.boxplot(x='Year', y='PayloadMass', data=df)
plt.xticks(rotation=45)
plt.show()

In [ ]:
#The drop observed in 2019 is not primarily due to lower payload values, as the distribution remains relatively strong.
#Instead, it is likely influenced by aggregation effects such as mean calculation and the number of launches in that year

In [ ]:
sns.countplot(x='Orbit', hue='Reused', data=df)
# Success rate by Orbit type
#we can observe vleo so Geo have high success rate

In [ ]:
sns.scatterplot(x='FlightNumber', y='PayloadMass', hue='Orbit', data=df)
#SpaceX getting heavier over time

In [ ]:
sns.countplot(x='LaunchSite', hue='Outcome', data=df)
plt.xticks(rotation=45)

In [ ]:
df2.info()

# Layer 2- Predictive Model

In [ ]:
df2['LandingSuccess'] = df2['Outcome'].apply(lambda x: 1 if str(x).startswith('True') else 0)
# We convert Outcome into a simple format:
# 1 = Success (True)
# 0 = Failure (False)
print(df2['LandingSuccess'].value_counts())

In [ ]:
df2.info()

In [ ]:
df2 = df2.drop(['Unnamed: 0', 'Outcome', 'LandingPad', 'Serial', 'Date'], axis = 1) #droping useless columns for model performance

In [ ]:
df2.info()

In [ ]:
X = df2.drop('LandingSuccess', axis=1)
y = df2['LandingSuccess']
X

In [ ]:
X


In [ ]:
y

In [ ]:
#so basically we gave the landingSuccess column to y, and gave other column to X. WHY? coz we are going to predict the landing Success
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
y_train

In [ ]:
y_test

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred_lr = lr.predict(X_test)

y_pred_lr

In [ ]:
y_test

In [ ]:

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth = 30, random_state=42)

rf.fit(X_train, y_train)

In [ ]:
y_pred_rf = rf.predict(X_test)

y_pred_rf

In [ ]:
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_rf))

In [ ]:
lr_acc = accuracy_score(y_test, y_pred_lr)
rf_acc = accuracy_score(y_test, y_pred_rf)

print("=== Model Comparison ===\n")
print(f"Logistic Regression Accuracy: {lr_acc}")
print(f"Random Forest Accuracy: {rf_acc}")

# Lets Observe

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm = confusion_matrix(y_test, y_pred_rf)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()

plt.title("Prediction vs Actual (Confusion Matrix)")
plt.show()

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(y_test.values, label="Actual", marker='o')
plt.plot(y_pred_rf, label="Predicted", marker='x')

plt.title("Actual vs Predicted Outcomes")
plt.xlabel("Test Sample Index")
plt.ylabel("Outcome (0 = Fail, 1 = Success)")
plt.legend()

plt.show()

In [ ]:

sample_index = 0  # you can change this

sample_data = X_test.iloc[sample_index]
actual = y_test.iloc[sample_index]

print("\nReal Launch Example")
print("-------------------")

for col, val in sample_data.items():
    print(f"{col}: {val}")

print("\nActual Outcome:", "Landed" if actual == 1 else "Failed")

# Simulation

In [ ]:
#Lets see which features are important
importance = pd.Series(rf.feature_importances_, index=X.columns)
importance = importance.sort_values(ascending=False)

print(importance)
len(importance)

In [ ]:
important_features = importance.head(7).index.tolist()
print(important_features)
df2 = df2[important_features + ['LandingSuccess']]
df2

In [ ]:
#Retrain with imp features only

X = df2.drop('LandingSuccess', axis=1)
y = df2['LandingSuccess']

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

In [ ]:
rf_ = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

lr_acc = accuracy_score(y_test, y_pred_lr)
rf_acc = accuracy_score(y_test, y_pred_rf)

print("\n=== Model Comparison ===")
print(f"Logistic Regression Accuracy: {lr_acc}")
print(f"Random Forest Accuracy: {rf_acc}")

In [ ]:
if rf_acc > lr_acc:
    final_model = rf
    print("\nUsing Random Forest as final model")
else:
    final_model = lr
    print("\nUsing Logistic Regression as final model")

#Product

In [ ]:
ranges = {
    'FlightNumber': (df['FlightNumber'].min(), df['FlightNumber'].max()),
    'PayloadMass': (df['PayloadMass'].min(), df['PayloadMass'].max()),
    'Block': (df['Block'].min(), df['Block'].max()),
    'Flights': (df['Flights'].min(), df['Flights'].max())
}


In [ ]:
#run this cell in colab..
'''for key, val in ranges.items():
    print(f"{key}: {val[0]} → {val[1]}")

user_input = {}

for feature in important_features:

    # ========================
    # Numerical with range
    # ========================
    if feature in ranges:
        min_val, max_val = ranges[feature]

        while True:
            user_val = input(f"Enter {feature} ({min_val} to {max_val}): ").strip()

            if user_val == "":
                print("⚠️ Input cannot be empty.")
                continue

            try:
                val = float(user_val)
                if val < min_val or val > max_val:
                    print("⚠️ Value out of range.")
                    continue
                break
            except ValueError:
                print("⚠️ Enter a valid number.")

        user_input[feature] = val

    # ========================
    # Boolean
    # ========================
    elif feature in ["GridFins", "Reused", "Legs"]:
        while True:
            user_val = input(f"{feature} (1 = Yes, 0 = No): ").strip()

            if user_val in ["0", "1"]:
                val = int(user_val)
                break
            else:
                print("⚠️ Enter 1 or 0 only.")

        user_input[feature] = val

    # ========================
    # Orbit
    # ========================
    elif "Orbit" in feature:
        print("\nSelect Orbit Type:")
        print("1 = GTO | 2 = LEO | 3 = ISS | 4 = Other")

        while True:
            user_val = input("Enter choice: ").strip()
            if user_val in ["1", "2", "3", "4"]:
                choice = int(user_val)
                break
            else:
                print("⚠️ Invalid choice.")

        # reset orbit columns
        for f in important_features:
            if "Orbit" in f:
                user_input[f] = 0

        if choice == 1 and "Orbit_GTO" in important_features:
            user_input["Orbit_GTO"] = 1
        elif choice == 2 and "Orbit_LEO" in important_features:
            user_input["Orbit_LEO"] = 1
        elif choice == 3 and "Orbit_ISS" in important_features:
            user_input["Orbit_ISS"] = 1

    # ========================
    # Other numeric features
    # ========================
    else:
        while True:
            user_val = input(f"Enter {feature}: ").strip()

            if user_val == "":
                print("⚠️ Input cannot be empty.")
                continue

            try:
                val = float(user_val)
                break
            except ValueError:
                print("⚠️ Enter a valid number.")

        user_input[feature] = val


# ========================
# DataFrame + Prediction
# ========================
input_df = pd.DataFrame([user_input])

# VERY IMPORTANT (don’t skip)
input_df = input_df[important_features]

print("\nAnalyzing launch parameters...")
time.sleep(2)

prediction = final_model.predict(input_df)
prob = final_model.predict_proba(input_df)

print("\n=== 🚀 Launch Prediction Result ===")

if prediction[0] == 1:
    print("✅ SUCCESS: Booster is likely to LAND successfully!")
else:
    print("❌ FAILURE: Booster is likely to FAIL landing.")

print(f"\nConfidence: {prob[0][1]*100:.2f}%")'''